# Fine-tune LLM using LoRA and QLoRA

In [8]:
%cd /content
!rm -rf LoRA-QLoRA-studies
!git clone https://github.com/shosakaue0808/LoRA-QLoRA-studies.git
%cd /content/LoRA-QLoRA-studies

/content
Cloning into 'LoRA-QLoRA-studies'...
remote: Enumerating objects: 130, done.
remote: Counting objects: 100% (130/130), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 130 (delta 68), reused 71 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (130/130), 199.78 KiB | 12.49 MiB/s, done.
Resolving deltas: 100% (68/68), done.
/content/LoRA-QLoRA-studies


In [9]:
!pip install transformers peft bitsandbytes python-dotenv datasets

In [14]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import prepare_model_for_kbit_training
from huggingface_hub import login
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from dotenv import load_dotenv
import os
# my files
import importlib

import src.train_utils as utils
import src.data_prep as data_prep
import src.lora_layer as lora

utils = importlib.reload(utils)
data_prep=importlib.reload(data_prep)
lora = importlib.reload(lora)

## login Hugging face

In [15]:
load_dotenv()
token = os.getenv("HF_TOKEN")
login(token)

In [16]:
# Set device (use GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


# Load Model

# LoRA base

In [17]:
model_id = "meta-llama/Llama-3.2-1B"

In [18]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
base_model_lora = AutoModelForCausalLM.from_pretrained(model_id)
base_model_lora.store_bit = "16bit"
tokenizer.pad_token = tokenizer.eos_token
base_model_lora.config.pad_token_id = tokenizer.pad_token_id

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

# QLoRA base

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
base_model_qlora = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config)
base_model_qlora.model_name = model_id
base_tokenizer_qlora = AutoTokenizer.from_pretrained(model_id)
base_tokenizer_qlora.pad_token = base_tokenizer_qlora.eos_token
base_model_qlora.config.pad_token_id = base_tokenizer_qlora.pad_token_id

In [ ]:
for name, p in base_model_lora.named_parameters():
    if p.requires_grad:
        print(name, p.shape)

In [19]:
#Set all weights freeze
for param in base_model_lora.parameters():
    param.requires_grad = False

# Attach LoRA layer to all linear layers in base model

In [ ]:
print(base_model_lora.model.layers)

In [20]:
lora.attach_Lora_to_Linear(base_model_lora, rank=16, alpha=16)

In [ ]:
print(base_model_lora)

In [ ]:
for name, p in base_model_lora.named_parameters():
    if p.requires_grad:
        print(name, p.shape)

In [21]:
# ratio of trainable / total
total_trainable_params = sum(p.numel() for p in base_model_lora.parameters() if p.requires_grad)
total_prams= sum(p.numel() for p in base_model_lora.parameters())
print(total_trainable_params/total_prams)

0.009038820617838861


# Load dataset

In [22]:
ds = load_dataset("openai/gsm8k", "main")

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [23]:
train_ds = ds["train"]
test_ds = ds["test"]

In [24]:
train_dataset = data_prep.GSMDataset(train_ds, tokenizer)
test_dataset = data_prep.GSMDataset(test_ds, tokenizer)

Max tokens: 460
Max tokens: 416


In [25]:
from torch.utils.data import random_split

train_size = int(0.9 * len(train_dataset))
val_size = len(train_dataset) - train_size

train_dataset, val_dataset = random_split(
    train_dataset,
    [train_size, val_size]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False
)

In [26]:
print(len(train_loader), len(val_loader), len(test_loader))

6725 748 1319


# Fine-tune process

In [27]:
base_model_lora = base_model_lora.to(device)
learning_rate = 1e-4
optimizer = torch.optim.AdamW(base_model_lora.parameters(), lr=learning_rate)
num_epochs = 2

In [28]:
train(base_model_lora, optimizer, train_loader, val_loader, num_epochs, device)

epoch 1 | step 200 | train_loss 0.5863 | val_loss 0.7686


AttributeError: 'LlamaForCausalLM' object has no attribute 'model_name'